# عريس الجنة — Sayed's voice (Egyptian Arabic)
Creates the real voice files for **Sayed only** with the free VoiceTut-TTS model. Nothing is paid; nothing in your game is changed.

**How to use:** click the ▶ play button on the left of each cell, **one at a time, top to bottom**. Wait for a green ✔ (or a printed "OK") before the next one.
If any cell prints **STOP**, copy that text and send it to me.

## Step 1 — Check the free GPU

In [ ]:
import subprocess, sys
try:
    import torch
except Exception as e:
    print("STOP: PyTorch is missing:", e); raise SystemExit
if not torch.cuda.is_available():
    print("STOP: no GPU. In the menu click Runtime > Change runtime type > choose T4 GPU > Save, then Runtime > Run all again.")
    print("If T4 is not offered, Google's free GPU quota is used up for now: try again later (free quota resets). Exact blocker: torch.cuda.is_available() == False")
    raise SystemExit
name = torch.cuda.get_device_name(0)
print("OK GPU:", name, "| torch", torch.__version__)
if "T4" not in name: print("Note: GPU is", name, "(not T4) - that is fine.")

## Step 2 — Install the tools (takes about 3-6 minutes)

In [ ]:
import subprocess, sys
def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print("STOP: command failed:", cmd); print(r.stdout[-1500:]); print(r.stderr[-2500:]); raise SystemExit
    return r.stdout
run("pip install -q git+https://github.com/k2-fsa/OmniVoice.git")
run("pip install -q voicetut-tts soundfile")
import voicetut_tts
print("OK installed voicetut-tts")

## Step 3 — Check the licences

In [ ]:
from huggingface_hub import hf_hub_download
import re, subprocess
card = open(hf_hub_download("mohammedaly22/VoiceTut-TTS", "README.md"), encoding="utf-8").read()
lic = re.search(r"^license:\s*(\S+)", card, re.M)
lic = lic.group(1) if lic else "UNKNOWN"
print("Model licence (Hugging Face card):", lic)
for pkg in ("voicetut-tts", "omnivoice"):
    out = subprocess.run(["pip", "show", pkg], capture_output=True, text=True).stdout
    l = [x for x in out.splitlines() if x.lower().startswith(("license", "name", "version"))]
    print(pkg, "->", " | ".join(l) if l else "not found")
if lic.lower() != "apache-2.0":
    print("STOP: model licence is not Apache-2.0 (it is", lic, "). Do not use the files until we review it."); raise SystemExit
print("OK: Apache-2.0 allows commercial use, with attribution.")
print("Notes: (1) the model card says do not use it to impersonate real people - we use a built-in studio voice, no cloning.")
print("       (2) the card does not state a separate licence for the 19 built-in voice reference clips; keep credit to VoiceTut-TTS / Mohammed Aly in your game credits.")

## Step 4 — Prepare the files (the same generator that is in your project)

In [ ]:
import os, json
os.makedirs("/content/arees/tools/voice-generator", exist_ok=True)
GEN = "#!/usr/bin/env python3\n\"\"\"Generate real Egyptian-Arabic WAV dialogue for AREES EL GANNAH with VoiceTut-TTS (free, local).\n\nSetup (once, on a machine with a GPU or a free Colab T4):\n    pip install voicetut-tts\n    pip install git+https://github.com/k2-fsa/OmniVoice.git\nRun from the project root:\n    python tools/voice-generator/generate.py --list-speakers\n    python tools/voice-generator/generate.py            # writes assets/audio/voices/**.wav + manifest.json\n    python tools/voice-generator/generate.py --only noga --force\n\nThe game reads assets/audio/voices/manifest.json at startup; missing clips silently fall back to\nbrowser speech / babble, so a partial run is safe.\n\"\"\"\nimport argparse, json, os, sys, wave\n\nROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), \"..\", \"..\"))\nOUT = os.path.join(ROOT, \"assets\", \"audio\", \"voices\")\nHERE = os.path.dirname(os.path.abspath(__file__))\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--list-speakers\", action=\"store_true\")\n    ap.add_argument(\"--only\", help=\"character id, e.g. sayed\")\n    ap.add_argument(\"--force\", action=\"store_true\", help=\"regenerate existing files\")\n    ap.add_argument(\"--model\", default=\"mohammedaly22/VoiceTut-TTS\")\n    ap.add_argument(\"--voices\", default=os.path.join(HERE, \"speakers.json\"))\n    a = ap.parse_args()\n    try:\n        from voicetut_tts import VoiceTutTTS\n    except ImportError:\n        sys.exit(\"voicetut-tts is not installed. See the header of this file / README.md.\")\n    tts = VoiceTutTTS.from_pretrained(a.model)\n    if a.list_speakers:\n        for s in tts.list_speakers(): print(s)\n        return\n    speakers = json.load(open(a.voices, encoding=\"utf-8\"))\n    lines = json.load(open(os.path.join(HERE, \"lines.json\"), encoding=\"utf-8\"))\n    manifest, made, skipped = {}, 0, 0\n    for ln in lines:\n        c, key = ln[\"char\"], ln[\"key\"]\n        if a.only and c != a.only: continue\n        path = os.path.join(OUT, c, key + \".wav\")\n        os.makedirs(os.path.dirname(path), exist_ok=True)\n        if os.path.exists(path) and not a.force: skipped += 1\n        else:\n            tts.synthesize(ln[\"text\"], speaker=speakers[c], output=path); made += 1\n            print(\"made\", os.path.relpath(path, ROOT))\n        manifest.setdefault(c, {})[key] = \"assets/audio/voices/%s/%s.wav\" % (c, key)\n    # validate every listed file is a readable WAV before publishing the manifest\n    for c, d in list(manifest.items()):\n        for k, p in list(d.items()):\n            try:\n                try:\n                    import soundfile as sf; i = sf.info(os.path.join(ROOT, p)); assert i.frames > 0\n                except ImportError:\n                    wave.open(os.path.join(ROOT, p)).close()\n            except Exception as e: print(\"BAD\", p, e); del d[k]\n    mp = os.path.join(OUT, \"manifest.json\")\n    if a.only and os.path.exists(mp):                      # merge partial runs\n        old = json.load(open(mp, encoding=\"utf-8\")); old.update(manifest); manifest = old\n    json.dump(manifest, open(mp, \"w\", encoding=\"utf-8\"), ensure_ascii=False, indent=1)\n    print(\"done: %d new, %d kept; manifest -> %s\" % (made, skipped, os.path.relpath(mp, ROOT)))\n\nif __name__ == \"__main__\":\n    main()\n"
LINES = "[\n {\n  \"char\": \"noga\",\n  \"key\": \"1\",\n  \"text\": \"قدامك ساعة، وبعدها هبعتلك الفرح في كيس.\"\n },\n {\n  \"char\": \"noga\",\n  \"key\": \"2\",\n  \"text\": \"سيد، أنا مش عايزة أعرف حصل إيه... أنا عايزة أعرف إنت عملت إيه تاني!\"\n },\n {\n  \"char\": \"shawky\",\n  \"key\": \"1\",\n  \"text\": \"أنا معاك يا سيد... بس من بعيد شوية.\"\n },\n {\n  \"char\": \"shawky\",\n  \"key\": \"2\",\n  \"text\": \"عندي خطة، بس محتاجة حد ينفذها غيري.\"\n },\n {\n  \"char\": \"kabsha\",\n  \"key\": \"1\",\n  \"text\": \"الفول عندي مش وجبة... ده مشروع قومي!\"\n },\n {\n  \"char\": \"kabsha\",\n  \"key\": \"2\",\n  \"text\": \"يا ابني إنت محتاج طاقة، مش محتاج دكتور نفسي!\"\n },\n {\n  \"char\": \"hankash\",\n  \"key\": \"1\",\n  \"text\": \"أنا حنكش... الاسم اللي بيقفل ملفات ويفتح مقابر!\"\n },\n {\n  \"char\": \"asaliya\",\n  \"key\": \"1\",\n  \"text\": \"اشرب يا سيد... الدنيا مش مستاهلة الضغط ده.\"\n },\n {\n  \"char\": \"asaliya\",\n  \"key\": \"2\",\n  \"text\": \"اللي يجري كتير، يعطش كتير.\"\n },\n {\n  \"char\": \"faheem\",\n  \"key\": \"1\",\n  \"text\": \"إنت سيد فلفل؟ معاك تصريح بالجري؟\"\n },\n {\n  \"char\": \"faheem\",\n  \"key\": \"2\",\n  \"text\": \"أنا شاكك فيك من ساعة ما شوفتك بتجري بشكل مريب!\"\n },\n {\n  \"char\": \"boga\",\n  \"key\": \"1\",\n  \"text\": \"تمام يا باشا... هنفذ الخطة زي ما حضرتك ما قلتهاش.\"\n },\n {\n  \"char\": \"boga\",\n  \"key\": \"2\",\n  \"text\": \"يا باشا، هو ينفع نأجل الشر لبكرة؟\"\n },\n {\n  \"char\": \"sayed\",\n  \"key\": \"intro\",\n  \"text\": \"الفرح ده هيتم، حتى لو اتجوزت في الإسعاف!\"\n },\n {\n  \"char\": \"sayed\",\n  \"key\": \"hit\",\n  \"text\": \"يا جماعة اهدوا، أنا مسيطر... تقريباً.\"\n },\n {\n  \"char\": \"sayed\",\n  \"key\": \"outro_lose\",\n  \"text\": \"هو أنا كل ما أعمل خير ألاقي نفسي في القسم؟\"\n },\n {\n  \"char\": \"sayed\",\n  \"key\": \"outro_win\",\n  \"text\": \"أنا مش متأخر... أنا بس بوصل في التوقيت الدرامي!\"\n },\n {\n  \"char\": \"sayed\",\n  \"key\": \"crate\",\n  \"text\": \"صندوق فيه مفاجأة! أحسن من مفاجآت الفرح!\"\n },\n {\n  \"char\": \"sayed\",\n  \"key\": \"chase\",\n  \"text\": \"بؤجة؟! ده راجل حنكش!\"\n },\n {\n  \"char\": \"sayed\",\n  \"key\": \"pw_mahshi\",\n  \"text\": \"حلة المحشي دي بقت خوذة رسمي!\"\n },\n {\n  \"char\": \"sayed\",\n  \"key\": \"pw_tray\",\n  \"text\": \"الشربات ده فيه صاروخ!\"\n },\n {\n  \"char\": \"sayed\",\n  \"key\": \"pw_ragab\",\n  \"text\": \"أم رجب باعتالي الحماية!\"\n },\n {\n  \"char\": \"noga\",\n  \"key\": \"intro\",\n  \"text\": \"قدامك ساعة، وبعدها هبعتلك الفرح في كيس.\"\n },\n {\n  \"char\": \"noga\",\n  \"key\": \"hit\",\n  \"text\": \"أنا اخترتك إنت، مش اخترت مسلسل الكوارث بتاعك!\"\n },\n {\n  \"char\": \"noga\",\n  \"key\": \"crate\",\n  \"text\": \"كسّر كمان! يمكن يطلع فيه فستان!\"\n },\n {\n  \"char\": \"noga\",\n  \"key\": \"chase\",\n  \"text\": \"بؤجة؟ حنكش باعت رجالته ورانا!\"\n },\n {\n  \"char\": \"noga\",\n  \"key\": \"pw_mahshi\",\n  \"text\": \"محدش يقرب مني دلوقتي!\"\n },\n {\n  \"char\": \"noga\",\n  \"key\": \"pw_tray\",\n  \"text\": \"وسّعوا... أنا جاية!\"\n },\n {\n  \"char\": \"noga\",\n  \"key\": \"pw_ragab\",\n  \"text\": \"تسلم إيدك يا أم رجب!\"\n },\n {\n  \"char\": \"shawky\",\n  \"key\": \"intro\",\n  \"text\": \"عندي خطة، بس محتاجة حد ينفذها غيري.\"\n },\n {\n  \"char\": \"shawky\",\n  \"key\": \"hit\",\n  \"text\": \"إحنا لو نجينا من دي، أنا هعتزل المصايب. مؤقتاً.\"\n },\n {\n  \"char\": \"shawky\",\n  \"key\": \"crate\",\n  \"text\": \"شكلها خطة! الصندوق ده كان جزء منها.\"\n },\n {\n  \"char\": \"shawky\",\n  \"key\": \"chase\",\n  \"text\": \"بؤجة! قلتلك فيه خطة... بس مش خطتي!\"\n },\n {\n  \"char\": \"shawky\",\n  \"key\": \"pw_mahshi\",\n  \"text\": \"أنا دلوقتي دبابة... دبابة محشي!\"\n },\n {\n  \"char\": \"shawky\",\n  \"key\": \"pw_tray\",\n  \"text\": \"ده مش جري... ده طيران!\"\n },\n {\n  \"char\": \"shawky\",\n  \"key\": \"pw_ragab\",\n  \"text\": \"أم رجب معانا؟ يبقى أنا شجاع!\"\n },\n {\n  \"char\": \"boga\",\n  \"key\": \"bogaChase\",\n  \"text\": \"الطلبات دي مش هتوصل الفرح... أوامر الباشا!\"\n },\n {\n  \"char\": \"boga\",\n  \"key\": \"bogaThrow\",\n  \"text\": \"خد عندك!\"\n },\n {\n  \"char\": \"boga\",\n  \"key\": \"bogaDizzy\",\n  \"text\": \"مين اللي حط قشر الموز ده هنا؟!\"\n },\n {\n  \"char\": \"boga\",\n  \"key\": \"bogaDefeat\",\n  \"text\": \"خلاص خلاص! هقول للباشا إنك كنت محظوظ!\"\n },\n {\n  \"char\": \"boga\",\n  \"key\": \"bogaHit1\",\n  \"text\": \"آآه! ده أنا كنت بهزر!\"\n },\n {\n  \"char\": \"boga\",\n  \"key\": \"bogaHit2\",\n  \"text\": \"يا باشا الحقني!\"\n },\n {\n  \"char\": \"boga\",\n  \"key\": \"bogaEnter\",\n  \"text\": \"حنكش باشا قال: مفيش فرح النهارده!\"\n },\n {\n  \"char\": \"boga\",\n  \"key\": \"bogaMiss\",\n  \"text\": \"هههه! فاتك القطر!\"\n },\n {\n  \"char\": \"hankash\",\n  \"key\": \"tease\",\n  \"text\": \"ماشي يا سيد فلفل... الجولة الجاية بتاعتي!\"\n }\n]"
open("/content/arees/tools/voice-generator/generate.py", "w", encoding="utf-8").write(GEN)
open("/content/arees/tools/voice-generator/lines.json", "w", encoding="utf-8").write(LINES)
print("OK files written;", len(json.loads(LINES)), "lines in file (only Sayed will be generated)")

## Step 5 — Choose Sayed's voice and generate (about 2-8 minutes)
Sayed uses the built-in male voice **Karim**. To try another, change `WANT = "Karim"` to Ahmed, Ali, Hassan, Mahmoud, Mohammed, Mostafa, Ramy, Shaker, Tarek, Yehia or Youssef, then run this cell again.

In [ ]:
import subprocess, json
WANT = "Karim"
r = subprocess.run(["voicetut", "--list-speakers"], capture_output=True, text=True)
names = [x.strip(" -*\t") for x in (r.stdout + r.stderr).splitlines() if x.strip()]
print("Available voices:", ", ".join(names))
match = [n for n in names if n.lower() == WANT.lower()]
if not match:
    print("STOP: voice", WANT, "not in list above. Change WANT to one of the names listed."); raise SystemExit
speakers = {c: "" for c in ["sayed","noga","shawky","hankash","boga","kabsha","asaliya","faheem"]}
speakers["sayed"] = match[0]
json.dump(speakers, open("/content/arees/tools/voice-generator/speakers.json", "w"))
p = subprocess.Popen(["python", "/content/arees/tools/voice-generator/generate.py", "--only", "sayed", "--force"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout: print(line, end="")
p.wait()
if p.returncode != 0: print("STOP: generator failed (see text above)"); raise SystemExit
print("OK generation finished")

## Step 6 — Check the files and listen
Press ▶ on each player below. **You should hear Sayed speaking Egyptian Arabic.** If a line is silent or wrong, tell me which one.

In [ ]:
import json, os, soundfile as sf, numpy as np
from IPython.display import Audio, display
root = "/content/arees"; lines = {l["key"]: l["text"] for l in json.loads(LINES) if l["char"] == "sayed"}
man = json.load(open(root + "/assets/audio/voices/manifest.json", encoding="utf-8")).get("sayed", {})
bad = []
for key, text in lines.items():
    path = root + "/" + man.get(key, "missing")
    if not os.path.isfile(path): bad.append((key, "file missing")); continue
    d, sr = sf.read(path); dur = len(d) / sr; rms = float(np.sqrt(np.mean(np.square(d))))
    ok = os.path.getsize(path) > 10000 and dur > 0.8 and rms > 0.005
    print(("OK  " if ok else "BAD ") + key.ljust(11), f"{dur:4.1f}s  {sr} Hz  loudness {rms:.3f}", "|", text)
    if not ok: bad.append((key, "too short/silent"))
    else: display(Audio(path))
print("\nSTOP: problem files:" if bad else "\nOK: all", len(lines), "files are valid." , bad if bad else "")

## Step 7 — Download
Run this cell. Your browser downloads **sayed_voice.zip**. Send that file to me (do not unzip it).

In [ ]:
import shutil, os
from google.colab import files
root = "/content/arees/assets/audio/voices"
tmp = "/content/pack/voices"; shutil.rmtree("/content/pack", ignore_errors=True); os.makedirs(tmp)
shutil.copytree(root + "/sayed", tmp + "/sayed"); shutil.copy(root + "/manifest.json", tmp + "/manifest.json")
shutil.make_archive("/content/sayed_voice", "zip", "/content/pack")
print("OK zip size:", round(os.path.getsize("/content/sayed_voice.zip") / 1024), "KB")
files.download("/content/sayed_voice.zip")